# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# .env file in this folder (project/starter) with:
# OPENAI_API_KEY="voc-..."
# OPENAI_BASE_URL="https://openai.vocareum.com/v1"
# TAVILY_API_KEY="tvly-..."

In [4]:
from dotenv import find_dotenv

load_dotenv(find_dotenv(), override=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

### VectorDB Instance

In [5]:
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [6]:
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)

In [7]:
# Recreate the collection so re-running the notebook doesn't keep stale data
if "udaplay" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection("udaplay")

collection = chroma_client.create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

### Add documents

In [8]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = (
        f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']} "
        f"Genre: {game['Genre']}. Publisher: {game['Publisher']}."
    )

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

print(f"Documents in collection: {collection.count()}")

Documents in collection: 15


### Semantic Search

In [9]:
results = collection.query(
    query_texts=["Who published Grand Theft Auto: San Andreas?"],
    n_results=3,
)

for doc_id, doc, meta, dist in zip(
    results["ids"][0], results["documents"][0], results["metadatas"][0], results["distances"][0]
):
    print(f"{doc_id} | distance={dist:.4f} | {meta['Name']} | {meta['Publisher']}")
    print(f"    {doc}")

002 | distance=0.1245 | Grand Theft Auto: San Andreas | Rockstar Games
    [PlayStation 2] Grand Theft Auto: San Andreas (2004) - An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson. Genre: Action-adventure. Publisher: Rockstar Games.
001 | distance=0.2110 | Gran Turismo | Sony Computer Entertainment
    [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre. Genre: Racing. Publisher: Sony Computer Entertainment.
003 | distance=0.2162 | Gran Turismo 5 | Sony Computer Entertainment
    [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics. Genre: Racing. Publisher: Sony Computer Entertainment.
